# Phase 2: Transaction-Level Data Cleaning & Preprocessing

**Project:** Customer Intelligence Engine (RFM Segmentation + 90-Day CLV Regression)  
**Input:** `data/raws/online_retail_II.xlsx` (Sheets: `Year 2009-2010` and `Year 2010-2011`)  

---

## Strict Scope & Interview Defense: Why We Stop at Transaction Level Here

This notebook **only** cleans, normalizes, audits, and partitions **transaction-level** records.

We **deliberately do NOT**:
1. **Compute RFM or any customer-level aggregation:** Computing RFM features before defining the temporal train/validation cutoff date ($T_{\text{cutoff}}$) causes **temporal data leakage** in 90-day CLV regression (future purchases leak into historical Recency, Frequency, and Monetary features). Customer aggregation belongs in a downstream Feature Engineering phase parameterized by the cutoff date.
2. **Apply scaling, log transforms, or categorical encoding:** Fit-transform operations (`StandardScaler`, `RobustScaler`, encoders) must be fit **strictly on the training split** after splitting, never on the full dataset.
3. **Train models:** Clean data separation ensures reproducible pipelines and independent unit testing.

### Allowed Derived Columns in This Notebook
- `source_sheet`: Origin Excel sheet name (needed to audit cross-sheet overlap)
- `is_cancellation`: Boolean indicator (`Invoice` starts with `"C"`)
- `TotalPrice`: `Quantity * Price`
- `invoice_date`, `year_month`: Simple calendar extracts for temporal quality checks

### Accounting Principle
Every row from the raw **1,067,371** rows will be accounted for in a **Row Accounting** table (`step | rows_before | rows_after | removed | reason`). Rather than silently deleting anomalous or non-standard records, we partition them into dedicated audit tables (`admin_rows`, `guest_transactions`, `returns_clean`, `anomalies`).

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

if "display" not in globals():
    try:
        from IPython.display import display
    except ImportError:
        display = print

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

# Resolve project root robustly regardless of working directory (root vs Notebooks/)
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raws" / "online_retail_II.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

assert RAW_DATA_PATH.exists(), f"Raw dataset not found at: {RAW_DATA_PATH}"

# Initialize Row Accounting log
row_accounting_log = []

def log_row_accounting(step: str, rows_before: int, rows_after: int, reason: str) -> pd.DataFrame:
    '''Append a step to the Row Accounting ledger and return the updated table.'''
    row_accounting_log.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "removed": rows_before - rows_after,
        "reason": reason,
    })
    return pd.DataFrame(row_accounting_log)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Raw Data Path: {RAW_DATA_PATH}")

Project Root : c:\Users\mosa\Desktop\Projects\Customer-Intelligence-Engine
Raw Data Path: data\raws\online_retail_II.xlsx


## Step 0: Load Both Sheets & Initialize Row Accounting

### Why We Do This First
1. **Why load both sheets explicitly?** `pd.read_excel(path)` defaults to `sheet_name=0`, silently dropping `Year 2010-2011` (~542k rows). For 90-day CLV and multi-year RFM, we need the full 24-month history.
2. **Why add `source_sheet` BEFORE `pd.concat`?** Once concatenated, row origin is lost. Knowing whether a row came from `Year 2009-2010` or `Year 2010-2011` is essential in **Step 2** when we investigate the date overlap between the two sheets.
3. **Why `ignore_index=True`?** Each sheet starts its row index at `0`. Without `ignore_index=True`, index labels `0..525460` would appear twice, causing ambiguous `.loc[]` indexing later.
4. **Why no filtering during load?** We preserve all raw rows first so our Row Accounting ledger starts from the exact raw ground-truth count.

In [6]:
with pd.ExcelFile(RAW_DATA_PATH) as xl:
    df_2009 = pd.read_excel(xl, sheet_name="Year 2009-2010")
    df_2009["source_sheet"] = "Year 2009-2010"

    df_2010 = pd.read_excel(xl, sheet_name="Year 2010-2011")
    df_2010["source_sheet"] = "Year 2010-2011"

df = pd.concat([df_2009, df_2010], ignore_index=True)
RAW_ROW_COUNT = len(df)

# Initialize Row Accounting table
accounting_df = log_row_accounting(
    step="Step 0: Load & Concat",
    rows_before=RAW_ROW_COUNT,
    rows_after=len(df),
    reason=f"Loaded Year 2009-2010 ({len(df_2009):,}) + Year 2010-2011 ({len(df_2010):,}) with source_sheet tag",
)

# Step 0 Summary
print("=" * 70)
print("STEP 0 SUMMARY: RAW LOAD")
print("=" * 70)
print(f"Sheet 1 (Year 2009-2010) shape: {df_2009.shape}")
print(f"Sheet 2 (Year 2010-2011) shape: {df_2010.shape}")
print(f"Combined raw DataFrame shape  : {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print("\nRaw Dtypes:")
print(df.dtypes)

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows (Random Seed = 42):")
display(df.sample(5, random_state=42))

STEP 0 SUMMARY: RAW LOAD
Sheet 1 (Year 2009-2010) shape: (525461, 9)
Sheet 2 (Year 2010-2011) shape: (541910, 9)
Combined raw DataFrame shape  : (1067371, 9)
Columns (9): ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']

Raw Dtypes:
Invoice                 object
StockCode               object
Description             object
Quantity                 int64
InvoiceDate     datetime64[us]
Price                  float64
Customer ID            float64
Country                    str
source_sheet               str
dtype: object

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."



5 Sample Rows (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
455941,532657,21314,SMALL GLASS HEART TRINKET POT,12,2010-11-14 11:10:00,2.10,14562.0,United Kingdom,Year 2009-2010
826291,563214,22383,LUNCH BAG SUKI DESIGN,2,2011-08-14 12:56:00,1.65,16370.0,United Kingdom,Year 2010-2011
191636,507597,22561,WOODEN SCHOOL COLOURING SET,12,2010-05-10 13:21:00,1.65,17700.0,United Kingdom,Year 2009-2010
25864,491634,21588,RETRO SPOT GIANT TUBE MATCHES,1,2009-12-11 15:40:00,2.55,17841.0,United Kingdom,Year 2009-2010
73233,496007,85232B,SET/3 RUSSIAN DOLL STACKING TINS,3,2010-01-28 12:32:00,4.95,15203.0,United Kingdom,Year 2009-2010


## Step 1: Type and Text Normalization

### Why Normalize Types & Text Before Auditing Overlap or Duplicates?
1. **Mixed Python types inside `object` columns:** Because Excel parses pure digits (`489434`, `85048`) as `int` and alphanumeric codes (`C489449`, `79323P`) as `str`, `Invoice` and `StockCode` currently mix `int` and `str` objects. That breaks sorting, exact matching, and Parquet serialization.
2. **Why pandas nullable `"string"` instead of `.astype(str)`:** Calling `.astype(str)` turns `NaN` into the literal string `"nan"` (which hides real missing values!). Furthermore, `Description` contains 4 numeric codes stored as `int`; calling `.str.strip()` on a mixed `object` column silently turns `int` values into `NaN`. Casting with `.astype("string")` safely converts those ints to strings while keeping genuine nulls as `<NA>`.
3. **Why strip whitespace & uppercase `StockCode`?** Trailing spaces in `StockCode` / `Description` or mixed casing (`79323p` vs `79323P`) cause the exact same product or identical rows to look different during deduplication (Step 3) and SKU pattern discovery (Step 4).
4. **Why cast `Customer ID` to nullable `Int64`?** Legacy NumPy integer arrays do not support `NaN`, forcing pandas to display customer IDs as floats (`14562.0`). Using pandas nullable `Int64` preserves missingness (`<NA>`) while storing clean integer IDs (`14562`).
5. **Why create `is_cancellation` now?** In this dataset, credit/cancellation invoices start with `'C'`. Flagging them explicitly (`Invoice.str.startswith("C")`) preserves the business signal before any filtering happens.

In [7]:
rows_before = len(df)

# --- Pre-normalization evidence checks ---
non_int_cust = (df["Customer ID"].dropna() % 1 != 0).sum()
sc_ws = (df["StockCode"].astype("string") != df["StockCode"].astype("string").str.strip()).sum()
sc_case = (df["StockCode"].astype("string").str.strip() != df["StockCode"].astype("string").str.strip().str.upper()).sum()
desc_ws = (df["Description"].astype("string").dropna() != df["Description"].astype("string").dropna().str.strip()).sum()
cnt_ws = (df["Country"].astype("string") != df["Country"].astype("string").str.strip()).sum()
inv_lower_c = df["Invoice"].astype("string").str.strip().str.startswith("c").sum()

print("Pre-Normalization Findings:")
print(f"  - Fractional Customer IDs          : {non_int_cust}")
print(f"  - StockCode whitespace anomalies   : {sc_ws:,}")
print(f"  - StockCode lowercase variations   : {sc_case:,}")
print(f"  - Description whitespace anomalies : {desc_ws:,}")
print(f"  - Country whitespace anomalies     : {cnt_ws:,}")
print(f"  - Lowercase 'c' Invoice prefixes   : {inv_lower_c:,}")

# --- Apply normalization ---
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Invoice"] = df["Invoice"].astype("string").str.strip()
df["StockCode"] = df["StockCode"].astype("string").str.strip().str.upper()
df["Description"] = df["Description"].astype("string").str.strip()
df["Country"] = df["Country"].astype("string").str.strip()
df["Customer ID"] = df["Customer ID"].astype("Int64")
df["source_sheet"] = df["source_sheet"].astype("string")

# Flag cancellations (Invoices starting with 'C')
df["is_cancellation"] = df["Invoice"].str.startswith("C").fillna(False)

# Update Row Accounting (no rows dropped in normalization)
accounting_df = log_row_accounting(
    step="Step 1: Type & Text Normalization",
    rows_before=rows_before,
    rows_after=len(df),
    reason="Cast dtypes (datetime, string, Int64), stripped text, uppercased StockCode, added is_cancellation",
)

# --- Step 1 Summary ---
print("\n" + "=" * 70)
print("STEP 1 SUMMARY: TYPE & TEXT NORMALIZATION")
print("=" * 70)
print(f"Shape: {df.shape}")
print("\nNormalized Dtypes:")
print(df.dtypes)

print("\nNull Counts After Normalization (verifying <NA> preservation):")
print(df.isna().sum())

print("\nis_cancellation Breakdown:")
print(df["is_cancellation"].value_counts())

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows (Same Random Seed = 42 for direct comparison with Step 0):")
display(df.sample(5, random_state=42))

Pre-Normalization Findings:
  - Fractional Customer IDs          : 0
  - StockCode whitespace anomalies   : 1
  - StockCode lowercase variations   : 3,471
  - Description whitespace anomalies : 213,035
  - Country whitespace anomalies     : 0
  - Lowercase 'c' Invoice prefixes   : 0

STEP 1 SUMMARY: TYPE & TEXT NORMALIZATION
Shape: (1067371, 10)

Normalized Dtypes:
Invoice                    string
StockCode                  string
Description                string
Quantity                    int64
InvoiceDate        datetime64[us]
Price                     float64
Customer ID                 Int64
Country                    string
source_sheet               string
is_cancellation           boolean
dtype: object

Null Counts After Normalization (verifying <NA> preservation):
Invoice                 0
StockCode               0
Description          4382
Quantity                0
InvoiceDate             0
Price                   0
Customer ID        243007
Country                 0
source

,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."



5 Sample Rows (Same Random Seed = 42 for direct comparison with Step 0):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
455941,532657,21314,SMALL GLASS HEART TRINKET POT,12,2010-11-14 11:10:00,2.10,14562,United Kingdom,Year 2009-2010,False
826291,563214,22383,LUNCH BAG SUKI DESIGN,2,2011-08-14 12:56:00,1.65,16370,United Kingdom,Year 2010-2011,False
191636,507597,22561,WOODEN SCHOOL COLOURING SET,12,2010-05-10 13:21:00,1.65,17700,United Kingdom,Year 2009-2010,False
25864,491634,21588,RETRO SPOT GIANT TUBE MATCHES,1,2009-12-11 15:40:00,2.55,17841,United Kingdom,Year 2009-2010,False
73233,496007,85232B,SET/3 RUSSIAN DOLL STACKING TINS,3,2010-01-28 12:32:00,4.95,15203,United Kingdom,Year 2009-2010,False


## Step 2: Sheet Overlap Investigation (Before Any Deduplication)

### Why This Overlap Exists & Why It Differs From Normal Duplicates
In the UCI Online Retail II workbook:
- Sheet 1 (`Year 2009-2010`) was extracted from **2009-12-01** through **2010-12-09**.
- Sheet 2 (`Year 2010-2011`) — which is also the classic standalone *Online Retail* dataset — was extracted from **2010-12-01** through **2011-12-09**.

Consequently, **December 1–9, 2010** appears in **both** sheets.

**Why this is fundamentally different from normal within-sheet duplicates:**
1. **Cross-sheet overlap** is an **ETL extraction / file-stitching artifact**: the exact same 9 days of order logs were exported twice into two Excel tabs. Keeping both copies artificially **doubles revenue, unit counts, and order frequency** for early December 2010, distorting Q4 seasonality and biasing RFM Frequency/Monetary features.
2. **Within-sheet duplicates** are identical lines appearing *within a single sheet export* (e.g., repeated line items inside a single invoice due to POS double-scans or separate basket entries within the same minute). They require a distinct business decision in **Step 3**.

If we ran `df.drop_duplicates()` blindly right now, we would conflate cross-sheet overlap with within-sheet duplicates (especially for any within-sheet duplicates that happen to fall inside Dec 1–9, 2010!). Therefore, we first audit the overlap using a **multiset comparison** (comparing exact row counts per unique tuple across both sheets) **without deleting anything yet**.

In [8]:
rows_before = len(df)

s1_mask = df["source_sheet"] == "Year 2009-2010"
s2_mask = df["source_sheet"] == "Year 2010-2011"

s1_min, s1_max = df.loc[s1_mask, "InvoiceDate"].min(), df.loc[s1_mask, "InvoiceDate"].max()
s2_min, s2_max = df.loc[s2_mask, "InvoiceDate"].min(), df.loc[s2_mask, "InvoiceDate"].max()

overlap_start = max(s1_min, s2_min)
overlap_end = min(s1_max, s2_max)

overlap_s1 = df[s1_mask & (df["InvoiceDate"] >= overlap_start) & (df["InvoiceDate"] <= overlap_end)]
overlap_s2 = df[s2_mask & (df["InvoiceDate"] >= overlap_start) & (df["InvoiceDate"] <= overlap_end)]

# Multiset comparison on all raw business columns (ignoring source_sheet)
compare_cols = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country", "is_cancellation"
]

c1 = overlap_s1.groupby(compare_cols, dropna=False).size().rename("count_sheet1")
c2 = overlap_s2.groupby(compare_cols, dropna=False).size().rename("count_sheet2")
overlap_comp = pd.concat([c1, c2], axis=1).fillna(0).astype(int)

identical_instances = int(overlap_comp[["count_sheet1", "count_sheet2"]].min(axis=1).sum())
only_sheet1_instances = int((overlap_comp["count_sheet1"] - overlap_comp["count_sheet2"]).clip(lower=0).sum())
only_sheet2_instances = int((overlap_comp["count_sheet2"] - overlap_comp["count_sheet1"]).clip(lower=0).sum())
differing_keys = overlap_comp[overlap_comp["count_sheet1"] != overlap_comp["count_sheet2"]]

# Also check within-sheet duplicates that happen inside the overlap window
within_s1_overlap_dups = int(overlap_s1.duplicated(subset=compare_cols).sum())
within_s2_overlap_dups = int(overlap_s2.duplicated(subset=compare_cols).sum())

# Update Row Accounting (investigation only — no rows removed yet)
accounting_df = log_row_accounting(
    step="Step 2: Sheet Overlap Audit",
    rows_before=rows_before,
    rows_after=len(df),
    reason=f"Audited overlap ({overlap_start.date()} to {overlap_end.date()}): {len(overlap_s1):,} rows per sheet, {len(differing_keys)} diffs",
)

print("=" * 70)
print("STEP 2 SUMMARY: CROSS-SHEET OVERLAP INVESTIGATION")
print("=" * 70)
print(f"Sheet 1 Date Range : {s1_min}  -->  {s1_max}")
print(f"Sheet 2 Date Range : {s2_min}  -->  {s2_max}")
print(f"Overlap Window     : {overlap_start}  -->  {overlap_end}")
print(f"\nRows in Overlap Window (Sheet 1): {len(overlap_s1):,}")
print(f"Rows in Overlap Window (Sheet 2): {len(overlap_s2):,}")
print(f"Identical row instances across both sheets : {identical_instances:,} (100.00%)")
print(f"Instances unique to Sheet 1 only           : {only_sheet1_instances:,}")
print(f"Instances unique to Sheet 2 only           : {only_sheet2_instances:,}")
print(f"Distinct row keys with differences         : {len(differing_keys):,}")
print(f"Within-sheet duplicate lines inside overlap: Sheet 1 = {within_s1_overlap_dups:,}, Sheet 2 = {within_s2_overlap_dups:,}")

assert len(differing_keys) == 0, "Unexpected mismatch between sheets in overlap window!"

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows from the Overlap Window (Sheet 1 vs Sheet 2):")
display(pd.concat([overlap_s1.head(3), overlap_s2.head(2)]))

STEP 2 SUMMARY: CROSS-SHEET OVERLAP INVESTIGATION
Sheet 1 Date Range : 2009-12-01 07:45:00  -->  2010-12-09 20:01:00
Sheet 2 Date Range : 2010-12-01 08:26:00  -->  2011-12-09 12:50:00
Overlap Window     : 2010-12-01 08:26:00  -->  2010-12-09 20:01:00

Rows in Overlap Window (Sheet 1): 22,523
Rows in Overlap Window (Sheet 2): 22,523
Identical row instances across both sheets : 22,523 (100.00%)
Instances unique to Sheet 1 only           : 0
Instances unique to Sheet 2 only           : 0
Distinct row keys with differences         : 0
Within-sheet duplicate lines inside overlap: Sheet 1 = 321, Sheet 2 = 321

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...



5 Sample Rows from the Overlap Window (Sheet 1 vs Sheet 2):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
502938,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Year 2009-2010,False
502939,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2009-2010,False
502940,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Year 2009-2010,False
525461,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Year 2010-2011,False
525462,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Year 2010-2011,False


## Step 3: Deduplication

### 3A. Remove Cross-Sheet Overlap Duplicates
Because Step 2 proved that all **22,523** overlapping rows (`2010-12-01 08:26` to `2010-12-09 20:01`) are **100% identical** between `Year 2009-2010` and `Year 2010-2011`, we safely drop the `Year 2009-2010` copy and keep the `Year 2010-2011` copy. This preserves the exact chronological continuity of Sheet 2 without double-counting early December 2010.

### 3B. Audit Exact Duplicates *Inside* a Single Sheet (Before Dropping!)
Before removing within-sheet duplicates, we must ask an important business question:
- **Are they POS / web-form double-scans** (accidental duplicate rows that inflate revenue and quantity)?
- **Or are they legitimate repeated basket lines** (because the storefront did not aggregate identical items added separately to a cart)?

To answer this empirically, we inspect:
1. Full-row duplicates vs. 4-key duplicates (`Invoice + StockCode + Quantity + Price`).
2. Whether duplicate rows appear **consecutively** (`index diff == 1`, typical of double-clicks/double-scans) or **scattered across the invoice** (`index diff > 1`, typical of browsing and re-adding items).
3. The distribution of `Quantity` and total monetary impact (`£`).

In [9]:
# --- 3A: Remove Cross-Sheet Overlap (keep Year 2010-2011 copy) ---
rows_before = len(df)
overlap_drop_mask = (
    (df["source_sheet"] == "Year 2009-2010")
    & (df["InvoiceDate"] >= overlap_start)
    & (df["InvoiceDate"] <= overlap_end)
)
removed_overlap = df[overlap_drop_mask].copy()
df = df[~overlap_drop_mask].copy()

accounting_df = log_row_accounting(
    step="Step 3A: Remove Cross-Sheet Overlap",
    rows_before=rows_before,
    rows_after=len(df),
    reason="Dropped Year 2009-2010 copy of identical Dec 1-9, 2010 overlap (kept Year 2010-2011)",
)

# --- 3B: Audit Within-Sheet Duplicates (Inspect Evidence Before Deciding) ---
full_dup_cols = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
key4_cols = ["Invoice", "StockCode", "Quantity", "Price"]

full_dup_all = df.duplicated(subset=full_dup_cols, keep=False)
full_dup_extra = df.duplicated(subset=full_dup_cols, keep="first")

key4_dup_all = df.duplicated(subset=key4_cols, keep=False)
key4_dup_extra = df.duplicated(subset=key4_cols, keep="first")

dups_df = df[full_dup_extra].copy()
dup_rev = (dups_df["Quantity"] * dups_df["Price"]).sum()
total_rev = (df["Quantity"] * df["Price"]).sum()

# Check row index spacing between identical lines
dup_spacing = df[full_dup_all].copy()
dup_spacing["orig_idx"] = dup_spacing.index
dup_spacing["idx_gap"] = dup_spacing.groupby(full_dup_cols, dropna=False)["orig_idx"].diff()
consecutive_count = int((dup_spacing["idx_gap"] == 1).sum())
non_consecutive_count = int((dup_spacing["idx_gap"] > 1).sum())

print("=" * 70)
print("STEP 3 SUMMARY: OVERLAP REMOVED & WITHIN-SHEET DUPLICATE AUDIT")
print("=" * 70)
print(f"Cross-sheet overlap rows removed       : {len(removed_overlap):,}")
print(f"Remaining rows after overlap removal   : {len(df):,}")
print(f"\nWithin-sheet full-row duplicates (extra): {int(full_dup_extra.sum()):,} ({full_dup_extra.mean()*100:.2f}%)")
print(f"Key4 (Inv+SKU+Qty+Price) duplicates     : {int(key4_dup_extra.sum()):,}")
print(f"Matches on Key4 differing elsewhere     : {int((key4_dup_all & ~full_dup_all).sum())} rows")
print(f"Total net revenue in extra duplicates   : £{dup_rev:,.2f} ({dup_rev / total_rev * 100:.2f}% of £{total_rev:,.2f})")
print(f"Index spacing -> Consecutive (gap == 1) : {consecutive_count:,} ({consecutive_count / len(dups_df) * 100:.1f}%)")
print(f"Index spacing -> Scattered   (gap > 1)  : {non_consecutive_count:,} ({non_consecutive_count / len(dups_df) * 100:.1f}%)")

print("\nTop 5 Quantities in Within-Sheet Duplicates:")
print(dups_df["Quantity"].value_counts().head(5))

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows of Key4 Matches Differing on Timestamp/Description:")
display(df[key4_dup_all & ~full_dup_all].sort_values(key4_cols).head(5))

print("\nSample Scattered Within-Sheet Duplicates (Invoice 489517):")
inv_489517 = df[df["Invoice"] == "489517"]
display(inv_489517[inv_489517.duplicated(subset=full_dup_cols, keep=False)].head(5))

print("\nSample Consecutive Within-Sheet Duplicates (Invoice 489556 & 489536):")
consec_invs = dup_spacing[dup_spacing["idx_gap"] == 1]["Invoice"].unique()[:2]
display(df[df["Invoice"].isin(consec_invs) & full_dup_all].head(5))

STEP 3 SUMMARY: OVERLAP REMOVED & WITHIN-SHEET DUPLICATE AUDIT
Cross-sheet overlap rows removed       : 22,523
Remaining rows after overlap removal   : 1,044,848

Within-sheet full-row duplicates (extra): 11,812 (1.13%)
Key4 (Inv+SKU+Qty+Price) duplicates     : 11,816
Matches on Key4 differing elsewhere     : 8 rows
Total net revenue in extra duplicates   : £54,228.42 (0.29% of £18,909,762.12)
Index spacing -> Consecutive (gap == 1) : 1,685 (14.3%)
Index spacing -> Scattered   (gap > 1)  : 10,127 (85.7%)

Top 5 Quantities in Within-Sheet Duplicates:
Quantity
1     8331
2     1303
6      533
12     475
3      279
Name: count, dtype: int64

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...



5 Sample Rows of Key4 Matches Differing on Timestamp/Description:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
41637,492807,21813,GARLAND WITH STARS AND BELLS,6,2009-12-20 12:28:00,4.95,17211,United Kingdom,Year 2009-2010,False
41657,492807,21813,GARLAND WITH STARS AND BELLS,6,2009-12-20 12:29:00,4.95,17211,United Kingdom,Year 2009-2010,False
724585,554084,23298,"BUNTING , SPOTTY",3,2011-05-22 11:52:00,4.95,12909,United Kingdom,Year 2010-2011,False
724605,554084,23298,SPOTTY BUNTING,3,2011-05-22 11:52:00,4.95,12909,United Kingdom,Year 2010-2011,False
871494,567183,22659,LUNCH BOX I LOVE LONDON,2,2011-09-18 15:32:00,1.95,14769,United Kingdom,Year 2010-2011,False



Sample Scattered Within-Sheet Duplicates (Invoice 489517):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,Year 2009-2010,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,Year 2009-2010,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom,Year 2009-2010,False
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom,Year 2009-2010,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom,Year 2009-2010,False



Sample Consecutive Within-Sheet Duplicates (Invoice 489556 & 489536):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
744,489536,21809,CHRISTMAS HANGING TREE WITH BELL,1,2009-12-01 12:13:00,1.25,16393,United Kingdom,Year 2009-2010,False
745,489536,21809,CHRISTMAS HANGING TREE WITH BELL,1,2009-12-01 12:13:00,1.25,16393,United Kingdom,Year 2009-2010,False
754,489536,85033S,SET/6 SILVER REINDEER T-LIGHTS,1,2009-12-01 12:13:00,1.95,16393,United Kingdom,Year 2009-2010,False
756,489536,22185,SLATE TILE NATURAL HANGING,2,2009-12-01 12:13:00,1.65,16393,United Kingdom,Year 2009-2010,False
757,489536,21786,RAIN HAT WITH RED SPOTS,1,2009-12-01 12:13:00,0.42,16393,United Kingdom,Year 2009-2010,False


### 3C. Decision Applied: Drop Exact Within-Sheet Duplicates
Following review of the audit evidence:
- All **11,812** exact full-row duplicates (`1.13%` of rows, `0.29%` of net revenue) are dropped keeping the first occurrence (`keep="first"`).
- The 4 `Key4` pairs that differed on timestamp (`1 min` apart) or updated description are preserved as separate records.
- *Known limitation documented for interview/README*: ~85.7% of exact duplicates were non-consecutive `Qty=1` re-entries (possibly separate unmerged web-cart clicks), so deduplication prevents accidental double-scans at the cost of at most `0.29%` of gross revenue.

In [10]:
rows_before = len(df)
df = df.drop_duplicates(subset=full_dup_cols, keep="first").copy()
within_sheet_dups_removed = rows_before - len(df)
TOTAL_REMOVED_DUPLICATES = len(removed_overlap) + within_sheet_dups_removed

accounting_df = log_row_accounting(
    step="Step 3B: Remove Within-Sheet Duplicates",
    rows_before=rows_before,
    rows_after=len(df),
    reason="Dropped exact full-row duplicates (keep='first'); kept 4 pairs differing in timestamp/description",
)

print("=" * 70)
print("STEP 3B SUMMARY: WITHIN-SHEET DEDUPLICATION APPLIED")
print("=" * 70)
print(f"Within-sheet exact duplicates removed : {within_sheet_dups_removed:,}")
print(f"Total duplicates removed (3A + 3B)    : {TOTAL_REMOVED_DUPLICATES:,}")
print(f"Remaining rows                        : {len(df):,}")

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows After Deduplication (Random Seed = 42):")
display(df.sample(5, random_state=42))

STEP 3B SUMMARY: WITHIN-SHEET DEDUPLICATION APPLIED
Within-sheet exact duplicates removed : 11,812
Total duplicates removed (3A + 3B)    : 34,335
Remaining rows                        : 1,033,036

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...



5 Sample Rows After Deduplication (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
228473,511497,20725,LUNCH BAG RED SPOTTY,12,2010-06-08 14:40:00,1.65,13089,United Kingdom,Year 2009-2010,False
1045079,580156,22193,RED DINER WALL CLOCK,4,2011-12-02 10:32:00,8.50,14897,United Kingdom,Year 2010-2011,False
908164,569943,22906,12 MESSAGE CARDS WITH ENVELOPES,1,2011-10-06 18:08:00,1.65,14592,United Kingdom,Year 2010-2011,False
707773,552536,23152,IVORY SWEETHEART WIRE LETTER RACK,4,2011-05-10 10:00:00,3.75,14209,United Kingdom,Year 2010-2011,False
402464,527950,22487,WHITE WOOD GARDEN PLANT LADDER,2,2010-10-19 15:33:00,9.95,16795,United Kingdom,Year 2009-2010,False


## Step 4: Non-Product Rows Discovery & Classification

### Why Discover Non-Product SKUs Directly From Data?
Standard product SKUs in this retailer's catalog follow a **5-digit integer pattern**, optionally followed by letter variants (e.g., `85123A`, `21314`) — regex `^\d{5}[A-Z]*$`.

Instead of copying a hardcoded list from a blog post (which risks missing dataset-specific codes like `CRUK`, `ADJUST2`, `B`, or `TEST001`, or accidentally dropping real products), we:
1. Extract **every** `StockCode` that does **not** match `^\d{5}[A-Z]*$`.
2. Inspect its row count, example `Description`, and total revenue (`Quantity * Price`).
3. Classify every discovered code into business categories (`postage`, `bank_charges`, `manual_adjustments`, `samples`, `test_codes`, `gift_vouchers`, `other_legacy_or_admin`) so we can decide **with evidence** which rows belong in `admin_rows`.

In [11]:
# Discover all StockCodes NOT matching standard 5-digit (+ optional letters) product pattern
normal_sku_pattern = r"^\d{5}[A-Z]*$"
non_normal_mask = ~df["StockCode"].str.match(normal_sku_pattern, na=False)
non_normal_df = df[non_normal_mask].copy()
non_normal_df["tmp_rev"] = non_normal_df["Quantity"] * non_normal_df["Price"]

def classify_non_normal_sku(code: str) -> str:
    if code in {"POST", "DOT", "C2"}:
        return "postage"
    if code in {"BANK CHARGES", "AMAZONFEE", "CRUK"}:
        return "bank_charges"
    if code in {"M", "D", "ADJUST", "ADJUST2", "B"}:
        return "manual_adjustments"
    if code == "S":
        return "samples"
    if code.startswith("TEST"):
        return "test_codes"
    if code.startswith("GIFT"):
        return "gift_vouchers"
    if code == "C3":
        return "other (unknown <NA> adjustment)"
    if code.startswith("DCGS") or code in {"SP1002", "PADS"}:
        return "other (legacy SKU / add-on product)"
    return "other"

non_normal_summary = (
    non_normal_df.groupby("StockCode", dropna=False)
    .agg(
        row_count=("Invoice", "count"),
        example_description=("Description", lambda s: s.dropna().mode().iloc[0] if len(s.dropna()) > 0 else "<NA>"),
        total_revenue=("tmp_rev", "sum"),
        valid_customer_rows=("Customer ID", lambda s: int(s.notna().sum())),
    )
    .reset_index()
)
non_normal_summary["category"] = non_normal_summary["StockCode"].map(classify_non_normal_sku)
non_normal_summary = non_normal_summary.sort_values(["category", "row_count"], ascending=[True, False]).reset_index(drop=True)

category_summary = (
    non_normal_summary.groupby("category")
    .agg(
        unique_codes=("StockCode", "count"),
        total_rows=("row_count", "sum"),
        total_revenue=("total_revenue", "sum"),
        valid_customer_rows=("valid_customer_rows", "sum"),
    )
    .sort_values("total_rows", ascending=False)
)

print("=" * 70)
print("STEP 4 DISCOVERY SUMMARY: NON-5-DIGIT STOCKCODES")
print("=" * 70)
print(f"Total rows not matching ^\\d{{5}}[A-Z]*$ : {len(non_normal_df):,}")
print(f"Unique non-standard StockCodes         : {non_normal_df['StockCode'].nunique()}")

print("\nCategory Breakdown:")
display(category_summary)

print("\nComplete Table of All 61 Discovered Non-Standard StockCodes:")
with pd.option_context("display.max_rows", 100):
    display(non_normal_summary)

STEP 4 DISCOVERY SUMMARY: NON-5-DIGIT STOCKCODES
Total rows not matching ^\d{5}[A-Z]*$ : 5,980
Unique non-standard StockCodes         : 61

Category Breakdown:


,unique_codes,total_rows,total_revenue,valid_customer_rows
category,,,,
postage,3,3788,433410.510,2253
manual_adjustments,5,1641,-236238.700,1319
other (legacy SKU / add-on product),36,179,1132.887,21
bank_charges,3,152,-264936.179,53
gift_vouchers,10,101,1686.610,0
samples,1,101,-5991.110,0
test_codes,2,17,203.500,16
other (unknown <NA> adjustment),1,1,0.000,0



Complete Table of All 61 Discovered Non-Standard StockCodes:


,StockCode,row_count,example_description,total_revenue,valid_customer_rows,category
0,BANK CHARGES,100,Bank Charges,-35482.249,37,bank_charges
1,AMAZONFEE,36,AMAZON FEE,-221520.500,0,bank_charges
2,CRUK,16,CRUK Commission,-7933.430,16,bank_charges
3,GIFT_0001_20,29,Dotcomgiftshop Gift Voucher £20.00,471.060,0,gift_vouchers
4,GIFT_0001_30,29,Dotcomgiftshop Gift Voucher £30.00,610.090,0,gift_vouchers
5,GIFT_0001_10,16,Dotcomgiftshop Gift Voucher £10.00,126.210,0,gift_vouchers
6,GIFT_0001_50,8,Dotcomgiftshop Gift Voucher £50.00,253.590,0,gift_vouchers
7,GIFT_0001_40,7,Dotcomgiftshop Gift Voucher £40.00,166.090,0,gift_vouchers
8,GIFT_0001_80,4,Dotcomgiftshop Gift Voucher £80.00,0.000,0,gift_vouchers
9,GIFT_0001_70,3,Dotcomgiftshop Gift Voucher £70.00,59.570,0,gift_vouchers


### 4B. Decision Applied: Move Non-Product & Administrative Rows to `admin_rows`
Based on our data discovery table above:
- **25 confirmed non-product/admin StockCodes (`5,801` rows)** — spanning `postage` (`POST`, `DOT`, `C2`), `manual_adjustments` (`M`, `D`, `ADJUST`, `ADJUST2`, `B`), `bank_charges` (`BANK CHARGES`, `AMAZONFEE`, `CRUK`), `gift_vouchers` (`GIFT*`), `samples` (`S`), `test_codes` (`TEST001`, `TEST002`), and `C3` — are moved (not deleted) into `admin_rows`.
- **36 legacy/add-on physical product SKUs (`179` rows)** (`DCGS*`, `SP1002`, `PADS`) are retained for transactional splitting in Steps 5 and 6 (note: `Price == 0` lines will automatically go to `anomalies` in Step 5, and `DCGS*` guest rows will go to `guest_transactions` in Step 6).

In [12]:
rows_before = len(df)

legacy_sku_mask = df["StockCode"].str.startswith("DCGS") | df["StockCode"].isin(["SP1002", "PADS"])
admin_mask = (~df["StockCode"].str.match(normal_sku_pattern, na=False)) & (~legacy_sku_mask)

admin_rows = df[admin_mask].copy()
admin_rows["admin_category"] = admin_rows["StockCode"].map(classify_non_normal_sku)
df = df[~admin_mask].copy()

accounting_df = log_row_accounting(
    step="Step 4B: Move Non-Product Rows to admin_rows",
    rows_before=rows_before,
    rows_after=len(df),
    reason=f"Moved {len(admin_rows):,} confirmed admin/non-product rows ({admin_rows['StockCode'].nunique()} codes) to admin_rows; kept {legacy_sku_mask.sum()} legacy product rows",
)

print("=" * 70)
print("STEP 4B SUMMARY: MOVED NON-PRODUCT ROWS TO admin_rows")
print("=" * 70)
print(f"Rows moved to admin_rows : {len(admin_rows):,}")
print(f"Remaining product rows   : {len(df):,}")

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows from admin_rows (Random Seed = 42):")
display(admin_rows.sample(5, random_state=42))

STEP 4B SUMMARY: MOVED NON-PRODUCT ROWS TO admin_rows
Rows moved to admin_rows : 5,801
Remaining product rows   : 1,027,235

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."



5 Sample Rows from admin_rows (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation,admin_category
678427,549717,BANK CHARGES,Bank Charges,1,2011-04-11 14:56:00,15.00,14606,United Kingdom,Year 2010-2011,False,bank_charges
182941,C506739,M,Manual,-1,2010-05-04 10:30:00,269.34,14911,EIRE,Year 2009-2010,True,manual_adjustments
143279,502980,DOT,DOTCOM POSTAGE,1,2010-03-29 14:55:00,18.26,<NA>,United Kingdom,Year 2009-2010,False,postage
274684,C516128,POST,POSTAGE,-1,2010-07-16 15:04:00,9.58,<NA>,United Kingdom,Year 2009-2010,True,postage
185736,C507057,M,Manual,-1,2010-05-06 10:04:00,396.20,12422,Australia,Year 2009-2010,True,manual_adjustments


## Step 5: Split Into `sales`, `returns`, and `anomalies`

### Why Order Matters (Split BEFORE Filtering `Quantity > 0`)
If we naively filtered `Quantity > 0` right at the start of cleaning, we would **silently destroy all `returns`** (`Quantity < 0`) and lose the ability to:
1. Track customer return rates and net CLV (returns reduce real lifetime value and signal dissatisfaction/churn risk).
2. Reconcile our Row Accounting ledger to the raw 1,067,371 rows.
3. Audit why zero-price rows exist (`Price == 0` warehouse write-offs, damaged stock, or free promotional giveaways).

### Mutual-Exclusivity Partition Rules
Every row in the remaining `1,027,235` product rows is assigned to **exactly one** group:
- **`sales`**: `not is_cancellation` AND `Quantity > 0` AND `Price > 0`
- **`returns`**: `is_cancellation` AND `Quantity < 0` AND `Price > 0`
- **`anomalies`**: everything else (`Price <= 0`, `Quantity == 0`, or inconsistent sign combinations such as negative `Quantity` on a non-C invoice).

In [13]:
rows_before = len(df)

sales_mask = (~df["is_cancellation"]) & (df["Quantity"] > 0) & (df["Price"] > 0)
returns_mask = df["is_cancellation"] & (df["Quantity"] < 0) & (df["Price"] > 0)
anomalies_mask = ~(sales_mask | returns_mask)

# Strict assertions: mutually exclusive and collectively exhaustive
assert (sales_mask & returns_mask).sum() == 0, "Overlap detected between sales and returns!"
assert (sales_mask & anomalies_mask).sum() == 0, "Overlap detected between sales and anomalies!"
assert (returns_mask & anomalies_mask).sum() == 0, "Overlap detected between returns and anomalies!"
assert (sales_mask.sum() + returns_mask.sum() + anomalies_mask.sum()) == len(df), "Partition does not cover all rows!"

sales = df[sales_mask].copy()
returns = df[returns_mask].copy()
anomalies = df[anomalies_mask].copy()

# Verify unexpected sign edge cases explicitly
non_c_neg_qty = ((~df["is_cancellation"]) & (df["Quantity"] < 0) & (df["Price"] > 0)).sum()
c_pos_qty = (df["is_cancellation"] & (df["Quantity"] > 0)).sum()
neg_price_rows = (df["Price"] < 0).sum()
zero_qty_rows = (df["Quantity"] == 0).sum()

# Update Row Accounting (anomalies partitioned out; sales + returns advance to Step 6)
accounting_df = log_row_accounting(
    step="Step 5: Split Sales / Returns / Anomalies",
    rows_before=rows_before,
    rows_after=len(sales) + len(returns),
    reason=f"Partitioned into sales ({len(sales):,}) + returns ({len(returns):,}); moved {len(anomalies):,} anomalies out",
)

print("=" * 70)
print("STEP 5 SUMMARY: 3-GROUP PARTITION")
print("=" * 70)
print(f"sales     : {len(sales):,} rows ({len(sales)/rows_before*100:.2f}%)")
print(f"returns   : {len(returns):,} rows ({len(returns)/rows_before*100:.2f}%)")
print(f"anomalies : {len(anomalies):,} rows ({len(anomalies)/rows_before*100:.2f}%)")
print(f"Total check: {len(sales) + len(returns) + len(anomalies):,} == {rows_before:,} (ASSERTION PASSED)")

print("\nAnomalies Forensic Breakdown:")
print(f"  - Non-C invoice with Quantity < 0 & Price > 0 : {non_c_neg_qty}")
print(f"  - C-invoice with Quantity > 0                 : {c_pos_qty}")
print(f"  - Price < 0                                   : {neg_price_rows}")
print(f"  - Quantity == 0                               : {zero_qty_rows}")
print(f"  - Price == 0 & Quantity > 0 (giveaways/found) : {((anomalies['Price'] == 0) & (anomalies['Quantity'] > 0)).sum():,}")
print(f"  - Price == 0 & Quantity < 0 (damages/lost)    : {((anomalies['Price'] == 0) & (anomalies['Quantity'] < 0)).sum():,}")
print(f"  - Anomalies with registered Customer ID       : {anomalies['Customer ID'].notna().sum():,}")

print("\nTop 10 Descriptions Inside Anomalies:")
print(anomalies["Description"].value_counts(dropna=False).head(10))

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows from anomalies (Random Seed = 42):")
display(anomalies.sample(5, random_state=42))

print("\n5 Sample Rows from returns (Random Seed = 42):")
display(returns.sample(5, random_state=42))

print("\n5 Sample Rows from sales (Random Seed = 42):")
display(sales.sample(5, random_state=42))

STEP 5 SUMMARY: 3-GROUP PARTITION
sales     : 1,003,357 rows (97.68%)
returns   : 17,915 rows (1.74%)
anomalies : 5,963 rows (0.58%)
Total check: 1,027,235 == 1,027,235 (ASSERTION PASSED)

Anomalies Forensic Breakdown:
  - Non-C invoice with Quantity < 0 & Price > 0 : 0
  - C-invoice with Quantity > 0                 : 0
  - Price < 0                                   : 0
  - Quantity == 0                               : 0
  - Price == 0 & Quantity > 0 (giveaways/found) : 2,572
  - Price == 0 & Quantity < 0 (damages/lost)    : 3,391
  - Anomalies with registered Customer ID       : 61

Top 10 Descriptions Inside Anomalies:
Description
<NA>                     4239
check                     160
?                          90
damages                    83
damaged                    81
found                      28
missing                    27
sold as set on dotcom      20
Damaged                    17
adjustment                 16
Name: count, dtype: Int64

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."
6,Step 5: Split Sales / Returns / Anomalies,1027235,1021272,5963,"Partitioned into sales (1,003,357) + returns (..."



5 Sample Rows from anomalies (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
83374,497143,84845C,<NA>,-12,2010-02-05 15:46:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
437012,531187,44228,<NA>,48,2010-11-05 15:04:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
151396,503817,37410,<NA>,-55,2010-04-07 14:46:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
122605,501115,20688,<NA>,420,2010-03-12 14:05:00,0.0,<NA>,United Kingdom,Year 2009-2010,False
120698,500911,22035,<NA>,-100,2010-03-11 10:22:00,0.0,<NA>,United Kingdom,Year 2009-2010,False



5 Sample Rows from returns (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
180561,C506477,22138,BAKING SET 9 PIECE RETROSPOT,-6,2010-04-30 10:25:00,4.95,16652,United Kingdom,Year 2009-2010,True
599064,C542366,72802A,ROSE SCENT CANDLE IN JEWELLED BOX,-1,2011-01-27 12:54:00,3.81,13777,United Kingdom,Year 2010-2011,True
788699,C559952,21258,VICTORIAN SEWING BOX LARGE,-2,2011-07-14 10:57:00,6.95,17696,United Kingdom,Year 2010-2011,True
209844,C509624,22635,CHILDS BREAKFAST SET DOLLY GIRL,-4,2010-05-25 09:53:00,9.95,16303,United Kingdom,Year 2009-2010,True
326701,C521233,22423,REGENCY CAKESTAND 3 TIER,-1,2010-09-03 11:17:00,10.95,13093,United Kingdom,Year 2009-2010,True



5 Sample Rows from sales (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
956505,573585,23334,IVORY WICKER HEART SMALL,6,2011-10-31 14:41:00,1.25,<NA>,United Kingdom,Year 2010-2011,False
1030383,578996,23469,CARD HOLDER LOVE BIRD SMALL,4,2011-11-27 16:02:00,4.15,13991,United Kingdom,Year 2010-2011,False
713462,553013,22678,FRENCH BLUE METAL DOOR SIGN 3,1,2011-05-12 18:19:00,2.46,<NA>,United Kingdom,Year 2010-2011,False
108139,499769,84908A,PINK 5 PATCH FLOWER CUSHION COVER,24,2010-03-02 13:25:00,1.95,13694,United Kingdom,Year 2009-2010,False
604401,542906,85116,BLACK CANDELABRA T-LIGHT HOLDER,11,2011-02-01 15:12:00,1.63,<NA>,United Kingdom,Year 2010-2011,False


## Step 6: Missing Customer ID (Isolate Guest Transactions)

### Why Move Guest Checkouts Out Instead of Imputing or Deleting?
1. **Why they cannot stay in `sales` or `returns`:** Both RFM segmentation and 90-day CLV regression operate at the **customer level**. A row without a valid `Customer ID` (`<NA>` or `0`) cannot be linked across time. Imputing missing IDs with `0` or `"Guest"` is a catastrophic mistake because it collapses ~230k+ unrelated shoppers into **one fake super-customer** with thousands of orders and millions in spend!
2. **Why move to `guest_transactions` instead of deleting:**
   - Keeps our **Row Accounting** ledger 100% reconciled.
   - Preserves store-level demand metrics and lets us test **Sampling Bias / Non-Stationarity**: Does the guest checkout share stay constant across 2009–2011, or does it spike during holiday months (Q4) or shift across years?

In [14]:
rows_before = len(sales) + len(returns)

# Combine sales and returns temporarily for unified guest share & temporal reporting
sales_guest_mask = sales["Customer ID"].isna() | (sales["Customer ID"] == 0)
returns_guest_mask = returns["Customer ID"].isna() | (returns["Customer ID"] == 0)

# Build monthly guest vs registered audit BEFORE splitting
all_valid_tx = pd.concat(
    [
        sales.assign(_tx_group="sale", _is_guest=sales_guest_mask),
        returns.assign(_tx_group="return", _is_guest=returns_guest_mask),
    ],
    ignore_index=True,
)
all_valid_tx["_rev"] = all_valid_tx["Quantity"] * all_valid_tx["Price"]
all_valid_tx["_guest_rev"] = all_valid_tx["_rev"].where(all_valid_tx["_is_guest"], 0.0)
all_valid_tx["_ym"] = all_valid_tx["InvoiceDate"].dt.to_period("M").astype(str)

total_tx_rows = len(all_valid_tx)
total_tx_rev = all_valid_tx["_rev"].sum()
guest_rows_total = int(all_valid_tx["_is_guest"].sum())
guest_rev_total = float(all_valid_tx["_guest_rev"].sum())

# Sales-only guest metrics
sales_rev_total = float((sales["Quantity"] * sales["Price"]).sum())
sales_guest_rows = int(sales_guest_mask.sum())
sales_guest_rev = float((sales.loc[sales_guest_mask, "Quantity"] * sales.loc[sales_guest_mask, "Price"]).sum())

# Returns-only guest metrics
returns_guest_rows = int(returns_guest_mask.sum())

# Monthly trend of guest share (rows & gross sales revenue)
monthly_guest = (
    all_valid_tx[all_valid_tx["_tx_group"] == "sale"]
    .groupby("_ym")
    .agg(
        total_rows=("Invoice", "count"),
        guest_rows=("_is_guest", "sum"),
        total_sales_rev=("_rev", "sum"),
        guest_sales_rev=("_guest_rev", "sum"),
    )
    .reset_index()
)
monthly_guest["guest_row_pct"] = (monthly_guest["guest_rows"] / monthly_guest["total_rows"] * 100).round(2)
monthly_guest["guest_rev_pct"] = (monthly_guest["guest_sales_rev"] / monthly_guest["total_sales_rev"] * 100).round(2)

# Move guest rows out of sales and returns into guest_transactions
guest_sales = sales[sales_guest_mask].copy()
guest_sales["guest_group"] = "sale"
guest_returns = returns[returns_guest_mask].copy()
guest_returns["guest_group"] = "return"
guest_transactions = pd.concat([guest_sales, guest_returns], ignore_index=True)

sales = sales[~sales_guest_mask].copy()
returns = returns[~returns_guest_mask].copy()

# Update Row Accounting
accounting_df = log_row_accounting(
    step="Step 6: Isolate Missing Customer ID",
    rows_before=rows_before,
    rows_after=len(sales) + len(returns),
    reason=f"Moved {len(guest_transactions):,} guest rows ({sales_guest_rows:,} sales + {returns_guest_rows:,} returns) to guest_transactions",
)

print("=" * 70)
print("STEP 6 SUMMARY: MISSING CUSTOMER ID & GUEST SHARE")
print("=" * 70)
print(f"Guest rows moved to guest_transactions : {len(guest_transactions):,} ({guest_rows_total / total_tx_rows * 100:.2f}% of sales+returns)")
print(f"  - From sales                         : {sales_guest_rows:,} ({sales_guest_rows / (sales_guest_rows + len(sales)) * 100:.2f}% of sales rows)")
print(f"  - From returns                       : {returns_guest_rows:,} ({returns_guest_rows / (returns_guest_rows + len(returns)) * 100:.2f}% of returns rows)")
print(f"Guest share of sales revenue (£)       : £{sales_guest_rev:,.2f} / £{sales_rev_total:,.2f} ({sales_guest_rev / sales_rev_total * 100:.2f}%)")
print(f"Guest share of net revenue (sales+ret) : £{guest_rev_total:,.2f} / £{total_tx_rev:,.2f} ({guest_rev_total / total_tx_rev * 100:.2f}%)")
print(f"\nRemaining registered sales             : {len(sales):,}")
print(f"Remaining registered returns           : {len(returns):,}")

print("\nGuest Share Over Time (Monthly Sales Rows & Revenue %):")
with pd.option_context("display.max_rows", 30):
    display(monthly_guest[["_ym", "total_rows", "guest_rows", "guest_row_pct", "total_sales_rev", "guest_sales_rev", "guest_rev_pct"]])

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows from guest_transactions (Random Seed = 42):")
display(guest_transactions.sample(5, random_state=42))

print("\n5 Sample Rows from registered sales (Random Seed = 42):")
display(sales.sample(5, random_state=42))

STEP 6 SUMMARY: MISSING CUSTOMER ID & GUEST SHARE
Guest rows moved to guest_transactions : 227,089 (22.24% of sales+returns)
  - From sales                         : 226,761 (22.60% of sales rows)
  - From returns                       : 328 (1.83% of returns rows)
Guest share of sales revenue (£)       : £2,575,278.90 / £19,643,861.64 (13.11%)
Guest share of net revenue (sales+ret) : £2,568,806.36 / £18,927,399.07 (13.57%)

Remaining registered sales             : 776,596
Remaining registered returns           : 17,587

Guest Share Over Time (Monthly Sales Rows & Revenue %):


,_ym,total_rows,guest_rows,guest_row_pct,total_sales_rev,guest_sales_rev,guest_rev_pct
0,2009-12,43320,13112,30.27,797901.160,119985.09,15.04
1,2010-01,30178,8806,29.18,612297.422,78584.44,12.83
2,2010-02,27817,4857,17.46,537909.676,39972.10,7.43
3,2010-03,39610,7962,20.1,761705.981,95732.31,12.57
4,2010-04,32667,5946,18.2,646340.932,61213.50,9.47
5,2010-05,33226,5074,15.27,643492.030,50757.90,7.89
6,2010-06,38211,7616,19.93,696583.190,67773.79,9.73
7,2010-07,31943,5396,16.89,632994.320,51506.36,8.14
8,2010-08,31953,6005,18.79,674037.720,79476.55,11.79
9,2010-09,40415,6417,15.88,869243.121,63698.31,7.33



Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."
6,Step 5: Split Sales / Returns / Anomalies,1027235,1021272,5963,"Partitioned into sales (1,003,357) + returns (..."
7,Step 6: Isolate Missing Customer ID,1021272,794183,227089,"Moved 227,089 guest rows (226,761 sales + 328 ..."



5 Sample Rows from guest_transactions (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation,guest_group
115954,541104,82613C,"METAL SIGN,CUPCAKE SINGLE HOOK",4,2011-01-13 14:29:00,0.83,<NA>,United Kingdom,Year 2010-2011,False,sale
89660,534289,16238,PARTY TIME PENCIL ERASERS,1,2010-11-22 11:25:00,0.43,<NA>,United Kingdom,Year 2009-2010,False,sale
195652,571713,23002,TRAVEL CARD WALLET SKULLS,1,2011-10-18 16:58:00,0.83,<NA>,United Kingdom,Year 2010-2011,False,sale
193622,570690,47369B,BLUE GREEN EMBROIDERY COSMETIC BAG,1,2011-10-11 16:37:00,5.79,<NA>,United Kingdom,Year 2010-2011,False,sale
171498,560926,22178,VICTORIAN GLASS HANGING T-LIGHT,2,2011-07-22 09:20:00,2.46,<NA>,United Kingdom,Year 2010-2011,False,sale



5 Sample Rows from registered sales (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
574327,540479,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-01-07 15:19:00,4.95,14889,United Kingdom,Year 2010-2011,False
398857,527494,22795,SWEETHEART RECIPE BOOK STAND,4,2010-10-18 12:04:00,6.75,14156,EIRE,Year 2009-2010,False
603618,542835,22285,HANGING HEN ON NEST DECORATION,12,2011-02-01 11:30:00,1.65,13939,United Kingdom,Year 2010-2011,False
99417,498729,21534,DAIRY MAID LARGE MILK JUG,6,2010-02-22 15:39:00,4.95,17829,United Arab Emirates,Year 2009-2010,False
1035870,579455,22966,GINGERBREAD MAN COOKIE CUTTER,12,2011-11-29 14:14:00,1.25,15239,United Kingdom,Year 2010-2011,False


## Step 7: Quality Checks on `sales` (Evidence Gathering Before Any Action)

### Why Inspect Top Outliers, Nulls, Countries, and Time Gaps?
Now that `sales` (`776,596` rows) contains only positive-quantity, positive-price, registered-customer product lines, we must verify whether extreme values are **genuine B2B wholesale orders** or **fat-finger entry errors** that were immediately canceled.
1. **Top 20 by `Quantity` & `TotalPrice` vs. matching `returns`:** Checking if a matching cancellation (`Customer ID`, `StockCode`, `abs(Quantity)`) follows shortly after reveals whether an extreme order was an aborted mistake (which would distort `sales` extremes and gross Monetary CLV if unchecked).
2. **Remaining `Description` nulls & odd `Country` values:** Ensures no hidden non-standard categories (`Unspecified`, `European Community`) sneak into regional grouping unnoticed.
3. **Weekly invoice counts & multi-day gaps:** Distinguishes expected seasonal store closures (Christmas/Easter) from silent ETL extraction outages.

In [15]:
rows_before = len(sales) + len(returns)

# Helper to check whether a matching cancellation follows in `returns` (matching SKU + Qty + Price) or `admin_rows` (by TotalPrice)
def check_following_cancellations(top_df: pd.DataFrame, returns_df: pd.DataFrame, admin_df: pd.DataFrame) -> pd.DataFrame:
    matches = []
    admin_rev = admin_df["Quantity"] * admin_df["Price"]
    for _, row in top_df.iterrows():
        exact_m = returns_df[
            (returns_df["Customer ID"] == row["Customer ID"])
            & (returns_df["StockCode"] == row["StockCode"])
            & (returns_df["Quantity"].abs() == row["Quantity"])
            & (returns_df["Price"] == row["Price"])
            & (returns_df["InvoiceDate"] >= row["InvoiceDate"])
        ]
        if len(exact_m) > 0:
            first_m = exact_m.sort_values("InvoiceDate").iloc[0]
            delay = first_m["InvoiceDate"] - row["InvoiceDate"]
            matches.append(f"EXACT -> {first_m['Invoice']} (+{delay})")
            continue

        # Check admin_rows (e.g. Manual 'M' reversal matching exact -TotalPrice)
        admin_m = admin_df[
            (admin_df["Customer ID"] == row["Customer ID"])
            & ((admin_rev + row["_tmp_total"]).abs() < 1e-2)
            & (admin_df["InvoiceDate"] >= row["InvoiceDate"])
        ]
        if len(admin_m) > 0:
            first_a = admin_m.sort_values("InvoiceDate").iloc[0]
            delay = first_a["InvoiceDate"] - row["InvoiceDate"]
            matches.append(f"ADMIN_REV -> {first_a['Invoice']} ({first_a['StockCode']}, +{delay})")
            continue

        partial_m = returns_df[
            (returns_df["Customer ID"] == row["Customer ID"])
            & (returns_df["StockCode"] == row["StockCode"])
            & (returns_df["InvoiceDate"] >= row["InvoiceDate"])
        ]
        if len(partial_m) > 0:
            first_p = partial_m.sort_values("InvoiceDate").iloc[0]
            delay = first_p["InvoiceDate"] - row["InvoiceDate"]
            matches.append(f"PARTIAL -> {first_p['Invoice']} (qty {first_p['Quantity']} @ £{first_p['Price']}, +{delay})")
        else:
            matches.append("None (Legitimate wholesale)")
    out = top_df[["Invoice", "StockCode", "Description", "Quantity", "Price", "_tmp_total", "InvoiceDate", "Customer ID", "Country"]].copy()
    out["following_cancellation"] = matches
    return out

sales_check = sales.assign(_tmp_total=sales["Quantity"] * sales["Price"])
top20_qty = check_following_cancellations(sales_check.nlargest(20, "Quantity"), returns, admin_rows)
top20_tot = check_following_cancellations(sales_check.nlargest(20, "_tmp_total"), returns, admin_rows)

# Null descriptions, odd countries, and weekly invoice counts
null_desc_sales = int(sales["Description"].isna().sum())
odd_countries = ["Unspecified", "European Community", "Channel Islands", "West Indies", "RSA"]
odd_country_summary = pd.DataFrame([
    {
        "Country": c,
        "rows": int((sales["Country"] == c).sum()),
        "invoices": int(sales.loc[sales["Country"] == c, "Invoice"].nunique()),
        "customers": int(sales.loc[sales["Country"] == c, "Customer ID"].nunique()),
    }
    for c in odd_countries if (sales["Country"] == c).any()
])

weekly_invoices = sales.set_index("InvoiceDate").resample("W")["Invoice"].nunique()
unique_sale_dates = pd.Series(sales["InvoiceDate"].dt.date.unique()).sort_values().reset_index(drop=True)
day_gaps = pd.to_datetime(unique_sale_dates).diff().dt.days

# Update Row Accounting (audit only — nothing deleted yet)
accounting_df = log_row_accounting(
    step="Step 7: Quality Checks on sales",
    rows_before=rows_before,
    rows_after=len(sales) + len(returns),
    reason=f"Audited top-20 outliers, {null_desc_sales} null descriptions, odd countries, and {len(weekly_invoices)} weekly invoice counts",
)

print("=" * 70)
print("STEP 7 SUMMARY: QUALITY CHECKS ON SALES")
print("=" * 70)
print(f"Remaining null Descriptions in sales : {null_desc_sales}")
print(f"Unique Countries in sales            : {sales['Country'].nunique()}")
print(f"Weekly invoice counts (106 weeks)    : min={weekly_invoices.min()}, median={weekly_invoices.median():.1f}, max={weekly_invoices.max()}")

print("\nLowest 5 Weeks by Unique Sales Invoices:")
print(weekly_invoices.sort_values().head(5))

print("\nTop 5 Multi-Day Sales Gaps (Confirming Holiday Closures):")
for idx in day_gaps.nlargest(5).index:
    print(f"  {unique_sale_dates.iloc[idx-1]} --> {unique_sale_dates.iloc[idx]} ({int(day_gaps.iloc[idx])} days)")

print("\nOdd / Ambiguous Country Summary:")
display(odd_country_summary)

print("\nTop 20 Sales by Quantity (with Cancellation Lookup):")
display(top20_qty)

print("\nTop 20 Sales by TotalPrice (with Cancellation Lookup):")
display(top20_tot)

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Sales Rows (Random Seed = 42):")
display(sales.sample(5, random_state=42))

STEP 7 SUMMARY: QUALITY CHECKS ON SALES
Remaining null Descriptions in sales : 0
Unique Countries in sales            : 41
Weekly invoice counts (106 weeks)    : min=0, median=315.5, max=667

Lowest 5 Weeks by Unique Sales Invoices:
InvoiceDate
2010-01-03      0
2011-01-02      0
2009-12-27     64
2010-12-26    107
2010-01-10    187
Name: Invoice, dtype: int64

Top 5 Multi-Day Sales Gaps (Confirming Holiday Closures):
  2009-12-23 --> 2010-01-04 (12 days)
  2010-12-23 --> 2011-01-04 (12 days)
  2010-04-01 --> 2010-04-06 (5 days)
  2011-04-21 --> 2011-04-26 (5 days)
  2011-04-28 --> 2011-05-01 (3 days)

Odd / Ambiguous Country Summary:


,Country,rows,invoices,customers
0,Unspecified,518,16,6
1,European Community,57,3,1
2,Channel Islands,1544,54,13
3,West Indies,54,1,1
4,RSA,122,2,2



Top 20 Sales by Quantity (with Cancellation Lookup):


,Invoice,StockCode,Description,Quantity,Price,_tmp_total,InvoiceDate,Customer ID,Country,following_cancellation
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,2011-12-09 09:15:00,16446,United Kingdom,EXACT -> C581484 (+0 days 00:12:00)
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,12346,United Kingdom,EXACT -> C541433 (+0 days 00:16:00)
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,2010-02-15 11:57:00,13902,Denmark,None (Legitimate wholesale)
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,2010-03-17 13:09:00,13902,Denmark,None (Legitimate wholesale)
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,2010-03-17 13:09:00,13902,Denmark,None (Legitimate wholesale)
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,2010-03-17 13:09:00,13902,Denmark,None (Legitimate wholesale)
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,2010-03-17 13:09:00,13902,Denmark,None (Legitimate wholesale)
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940,United Kingdom,None (Legitimate wholesale)
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940,United Kingdom,None (Legitimate wholesale)
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940,United Kingdom,None (Legitimate wholesale)



Top 20 Sales by TotalPrice (with Cancellation Lookup):


,Invoice,StockCode,Description,Quantity,Price,_tmp_total,InvoiceDate,Customer ID,Country,following_cancellation
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,2011-12-09 09:15:00,16446,United Kingdom,EXACT -> C581484 (+0 days 00:12:00)
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,12346,United Kingdom,EXACT -> C541433 (+0 days 00:16:00)
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.00,2011-06-10 15:28:00,15098,United Kingdom,"ADMIN_REV -> C556445 (M, +0 days 00:03:00)"
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,15818.40,2010-11-04 11:36:00,15838,United Kingdom,"PARTIAL -> C536757 (qty -9360 @ £0.03, +28 day..."
228042,511465,15044A,PINK PAPER PARASOL,3500,2.55,8925.00,2010-06-08 12:59:00,18008,United Kingdom,None (Legitimate wholesale)
873786,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,5.06,7144.72,2011-09-20 11:05:00,17450,United Kingdom,None (Legitimate wholesale)
578172,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,2011-01-11 12:55:00,15749,United Kingdom,EXACT -> C550456 (+97 days 00:13:00)
686007,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,6539.40,2011-04-18 13:20:00,15749,United Kingdom,None (Legitimate wholesale)
379875,525968,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3120,1.66,5179.20,2010-10-08 10:10:00,15838,United Kingdom,"PARTIAL -> C536757 (qty -9360 @ £0.03, +55 day..."
947062,573003,23084,RABBIT NIGHT LIGHT,2400,2.08,4992.00,2011-10-27 12:11:00,14646,Netherlands,None (Legitimate wholesale)



Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."
6,Step 5: Split Sales / Returns / Anomalies,1027235,1021272,5963,"Partitioned into sales (1,003,357) + returns (..."
7,Step 6: Isolate Missing Customer ID,1021272,794183,227089,"Moved 227,089 guest rows (226,761 sales + 328 ..."
8,Step 7: Quality Checks on sales,794183,794183,0,"Audited top-20 outliers, 0 null descriptions, ..."



5 Sample Sales Rows (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
574327,540479,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-01-07 15:19:00,4.95,14889,United Kingdom,Year 2010-2011,False
398857,527494,22795,SWEETHEART RECIPE BOOK STAND,4,2010-10-18 12:04:00,6.75,14156,EIRE,Year 2009-2010,False
603618,542835,22285,HANGING HEN ON NEST DECORATION,12,2011-02-01 11:30:00,1.65,13939,United Kingdom,Year 2010-2011,False
99417,498729,21534,DAIRY MAID LARGE MILK JUG,6,2010-02-22 15:39:00,4.95,17829,United Arab Emirates,Year 2009-2010,False
1035870,579455,22966,GINGERBREAD MAN COOKIE CUTTER,12,2011-11-29 14:14:00,1.25,15239,United Kingdom,Year 2010-2011,False


### 7B. Decision Applied: Isolate 3 Same-Day Aborted Entry Errors (`5` Product Rows) to `anomalies`
Checking both `Quantity` and `Price` uncovered a critical detail on the 3rd mega-order (`556444` for `£38,970.00`):
- `581483` (`+80,995 @ £2.08 = £168,469.60`) was canceled 12 min later by `C581484` (`-80,995 @ £2.08`).
- `541431` (`+74,215 @ £1.04 = £77,183.60`) was canceled 16 min later by `C541433` (`-74,215 @ £1.04`).
- `556444` (`+60 @ £649.50 = £38,970.00`) was actually canceled 3 min later via a **Manual (`M`)** credit (`C556445`, `-£38,970.00`, already in `admin_rows`!), while `C556448` (`-60 @ £4.95 = -£297.00`) canceled an earlier order `556442` (`+60 @ £4.95`)!

Therefore, we move **only those 5 product rows** (`581483`, `C581484`, `541431`, `C541433`, `556444`) into `anomalies` and keep `C556448` inside `returns` so `556442` remains properly balanced. We also keep odd `Country` values (`Unspecified`, `European Community`) intact.

In [16]:
rows_before = len(sales) + len(returns)

aborted_sale_invoices = {"581483", "541431", "556444"}
aborted_return_invoices = {"C581484", "C541433"}  # C556445 is already in admin_rows; C556448 balances 556442

aborted_sales_mask = sales["Invoice"].isin(aborted_sale_invoices)
aborted_returns_mask = returns["Invoice"].isin(aborted_return_invoices)

assert aborted_sales_mask.sum() == 3, f"Expected 3 aborted sale rows, got {aborted_sales_mask.sum()}"
assert aborted_returns_mask.sum() == 2, f"Expected 2 aborted return rows, got {aborted_returns_mask.sum()}"

aborted_rows = pd.concat([sales[aborted_sales_mask], returns[aborted_returns_mask]], ignore_index=False)
anomalies = pd.concat([anomalies, aborted_rows], ignore_index=False)

sales = sales[~aborted_sales_mask].copy()
returns = returns[~aborted_returns_mask].copy()

accounting_df = log_row_accounting(
    step="Step 7B: Isolate Aborted Mega-Order Errors",
    rows_before=rows_before,
    rows_after=len(sales) + len(returns),
    reason="Moved 3 same-day (3-16 min) aborted entry errors + 2 product return counterparts (5 rows) to anomalies",
)

print("=" * 70)
print("STEP 7B SUMMARY: ABORTED ENTRY ERRORS MOVED TO anomalies")
print("=" * 70)
print(f"Rows moved to anomalies : {len(aborted_rows)} (total anomalies now: {len(anomalies):,})")
print(f"Remaining sales rows    : {len(sales):,} (max Qty={sales['Quantity'].max():,}, max Price=£{sales['Price'].max():,.2f})")
print(f"Remaining returns rows  : {len(returns):,}")

print("\nRow Accounting Table:")
display(accounting_df)

print("\nAll 5 Rows Moved to anomalies:")
display(aborted_rows)

STEP 7B SUMMARY: ABORTED ENTRY ERRORS MOVED TO anomalies
Rows moved to anomalies : 5 (total anomalies now: 5,968)
Remaining sales rows    : 776,593 (max Qty=19,152, max Price=£649.50)
Remaining returns rows  : 17,585

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."
6,Step 5: Split Sales / Returns / Anomalies,1027235,1021272,5963,"Partitioned into sales (1,003,357) + returns (..."
7,Step 6: Isolate Missing Customer ID,1021272,794183,227089,"Moved 227,089 guest rows (226,761 sales + 328 ..."
8,Step 7: Quality Checks on sales,794183,794183,0,"Audited top-20 outliers, 0 null descriptions, ..."
9,Step 7B: Isolate Aborted Mega-Order Errors,794183,794178,5,Moved 3 same-day (3-16 min) aborted entry erro...



All 5 Rows Moved to anomalies:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom,Year 2010-2011,False
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,United Kingdom,Year 2010-2011,False
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,Year 2010-2011,False
587085,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346,United Kingdom,Year 2010-2011,True
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446,United Kingdom,Year 2010-2011,True


## Step 8: Add Minimal Transaction-Level Columns

### Why Add Only `TotalPrice`, `invoice_date`, and `year_month`?
We deliberately restrict derived columns here to simple deterministic transaction-level attributes:
- `TotalPrice` (`Quantity * Price`): Standardizes line-level monetary spend across `sales`, `returns`, `guest_transactions`, `admin_rows`, and `anomalies`.
- `invoice_date` (`InvoiceDate` normalized to day precision): Enables fast daily grouping and cutoff filtering ($T_{\text{cutoff}}$) downstream without string parsing.
- `year_month` (`YYYY-MM` string): Enables clean monthly cohort/seasonality inspection.

No customer aggregations (RFM), scalers, or encodings are added here so downstream train/test temporal splits remain 100% leakage-free.

In [17]:
rows_before = len(sales) + len(returns)

def add_minimal_tx_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    df_out["TotalPrice"] = df_out["Quantity"] * df_out["Price"]
    df_out["invoice_date"] = df_out["InvoiceDate"].dt.normalize()
    df_out["year_month"] = df_out["InvoiceDate"].dt.to_period("M").astype("string")
    return df_out

sales = add_minimal_tx_columns(sales)
returns = add_minimal_tx_columns(returns)
guest_transactions = add_minimal_tx_columns(guest_transactions)
admin_rows = add_minimal_tx_columns(admin_rows)
anomalies = add_minimal_tx_columns(anomalies)

# Verify exact allowed column schema on sales and returns
expected_core_cols = [
    "Invoice", "StockCode", "Description", "Quantity", "InvoiceDate",
    "Price", "Customer ID", "Country", "source_sheet", "is_cancellation",
    "TotalPrice", "invoice_date", "year_month"
]
assert list(sales.columns) == expected_core_cols, f"Unexpected columns in sales: {list(sales.columns)}"
assert list(returns.columns) == expected_core_cols, f"Unexpected columns in returns: {list(returns.columns)}"

accounting_df = log_row_accounting(
    step="Step 8: Add Minimal Transaction Columns",
    rows_before=rows_before,
    rows_after=len(sales) + len(returns),
    reason="Added TotalPrice, invoice_date, and year_month to all partitioned tables",
)

print("=" * 70)
print("STEP 8 SUMMARY: MINIMAL TRANSACTION COLUMNS ADDED")
print("=" * 70)
print(f"sales shape   : {sales.shape}")
print(f"returns shape : {returns.shape}")
print(f"New max TotalPrice in sales : £{sales['TotalPrice'].max():,.2f}")
print(f"Total clean sales revenue   : £{sales['TotalPrice'].sum():,.2f}")
print(f"Total clean returns revenue : £{returns['TotalPrice'].sum():,.2f}")
print("\nFinal Column Dtypes (sales):")
print(sales.dtypes)

print("\nRow Accounting Table:")
display(accounting_df)

print("\n5 Sample Rows from sales (Random Seed = 42):")
display(sales.sample(5, random_state=42))

STEP 8 SUMMARY: MINIMAL TRANSACTION COLUMNS ADDED
sales shape   : (776593, 13)
returns shape : (17585, 13)
New max TotalPrice in sales : £15,818.40
Total clean sales revenue   : £16,783,959.54
Total clean returns revenue : £-464,336.83

Final Column Dtypes (sales):
Invoice                    string
StockCode                  string
Description                string
Quantity                    int64
InvoiceDate        datetime64[us]
Price                     float64
Customer ID                 Int64
Country                    string
source_sheet               string
is_cancellation           boolean
TotalPrice                float64
invoice_date       datetime64[us]
year_month                 string
dtype: object

Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."
6,Step 5: Split Sales / Returns / Anomalies,1027235,1021272,5963,"Partitioned into sales (1,003,357) + returns (..."
7,Step 6: Isolate Missing Customer ID,1021272,794183,227089,"Moved 227,089 guest rows (226,761 sales + 328 ..."
8,Step 7: Quality Checks on sales,794183,794183,0,"Audited top-20 outliers, 0 null descriptions, ..."
9,Step 7B: Isolate Aborted Mega-Order Errors,794183,794178,5,Moved 3 same-day (3-16 min) aborted entry erro...



5 Sample Rows from sales (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation,TotalPrice,invoice_date,year_month
1036963,579503,22699,ROSES REGENCY TEACUP AND SAUCER,4,2011-11-29 16:23:00,2.95,17097,Spain,Year 2010-2011,False,11.80,2011-11-29,2011-11
640350,546091,84978,HANGING HEART JAR T-LIGHT HOLDER,12,2011-03-09 11:53:00,1.25,14936,Channel Islands,Year 2010-2011,False,15.00,2011-03-09,2011-03
693008,550993,22384,LUNCH BAG PINK POLKADOT,10,2011-04-21 19:51:00,1.65,15810,United Kingdom,Year 2010-2011,False,16.50,2011-04-21,2011-04
221760,510993,21890,S/6 WOODEN SKITTLES IN COTTON BAG,6,2010-06-04 13:47:00,2.95,18206,United Kingdom,Year 2009-2010,False,17.70,2010-06-04,2010-06
303,489488,22086,PAPER CHAIN KIT 50'S CHRISTMAS,5,2009-12-01 10:59:00,2.95,17238,United Kingdom,Year 2009-2010,False,14.75,2009-12-01,2009-12


## Step 9: Loud Validation Assertions & Full Reconciliation

### Why Write Loud Assertions Before Exporting?
In production ML pipelines, silent data bugs (e.g., an unhandled duplicate, a stray cancellation inside `sales`, or a row-conservation mismatch) poison downstream RFM and CLV models without throwing errors. Writing explicit `assert` statements that **fail loudly** guarantees our cleaned outputs uphold every contract before writing Parquet files in Step 10.

In [18]:
rows_before = len(sales) + len(returns)

# 1. No duplicates in sales
assert sales.duplicated().sum() == 0, f"Found {sales.duplicated().sum()} full duplicates in sales!"
assert sales.duplicated(subset=full_dup_cols).sum() == 0, "Found duplicates on business columns in sales!"

# 2. Price > 0 and Quantity > 0 for all sales rows (and valid return invariants)
assert (sales["Price"] > 0).all(), f"Found {(sales['Price'] <= 0).sum()} non-positive Price rows in sales!"
assert (sales["Quantity"] > 0).all(), f"Found {(sales['Quantity'] <= 0).sum()} non-positive Quantity rows in sales!"
assert (sales["TotalPrice"] > 0).all(), "Found non-positive TotalPrice rows in sales!"
assert (returns["Price"] > 0).all(), "Found non-positive Price rows in returns!"
assert (returns["Quantity"] < 0).all(), "Found non-negative Quantity rows in returns!"

# 3. No null (or 0) Customer ID in sales and returns
assert sales["Customer ID"].notna().all(), f"Found {sales['Customer ID'].isna().sum()} null Customer IDs in sales!"
assert (sales["Customer ID"] != 0).all(), "Found Customer ID == 0 in sales!"
assert returns["Customer ID"].notna().all(), f"Found {returns['Customer ID'].isna().sum()} null Customer IDs in returns!"
assert (returns["Customer ID"] != 0).all(), "Found Customer ID == 0 in returns!"

# 4. No cancellation invoice inside sales (and only cancellations inside returns)
assert (~sales["is_cancellation"]).all(), "Found is_cancellation==True inside sales!"
assert (~sales["Invoice"].str.startswith("C")).all(), "Found C-prefix Invoice inside sales!"
assert returns["is_cancellation"].all(), "Found non-cancellation inside returns!"

# 5. Sum of all output groups + removed duplicates equals raw rows (1,067,371)
reconciled_total = (
    len(sales)
    + len(returns)
    + len(anomalies)
    + len(admin_rows)
    + len(guest_transactions)
    + TOTAL_REMOVED_DUPLICATES
)
assert reconciled_total == RAW_ROW_COUNT == 1_067_371, (
    f"Row conservation failed! Sum={reconciled_total:,} vs Raw={RAW_ROW_COUNT:,}"
)

# 6. Row Accounting ledger internal consistency & end-to-end reconciliation
for idx_row in range(len(accounting_df)):
    r = accounting_df.iloc[idx_row]
    assert r["rows_before"] - r["removed"] == r["rows_after"], f"Math mismatch at row {idx_row}: {r['step']}"
    if idx_row > 0:
        prev_after = accounting_df.iloc[idx_row - 1]["rows_after"]
        assert r["rows_before"] == prev_after, f"Continuity break at {r['step']}: {r['rows_before']} != {prev_after}"

assert RAW_ROW_COUNT - int(accounting_df["removed"].sum()) == len(sales) + len(returns), (
    "Row Accounting removed sum does not reconcile with final sales + returns!"
)

accounting_df = log_row_accounting(
    step="Step 9: Validation Assertions",
    rows_before=rows_before,
    rows_after=len(sales) + len(returns),
    reason="All 6 groups of loud data-contract & reconciliation assertions PASSED",
)

reconciliation_breakdown = pd.DataFrame([
    {"Group": "sales (clean registered)", "Rows": len(sales), "Share (%)": round(len(sales) / RAW_ROW_COUNT * 100, 2)},
    {"Group": "returns (clean registered)", "Rows": len(returns), "Share (%)": round(len(returns) / RAW_ROW_COUNT * 100, 2)},
    {"Group": "guest_transactions (missing Customer ID)", "Rows": len(guest_transactions), "Share (%)": round(len(guest_transactions) / RAW_ROW_COUNT * 100, 2)},
    {"Group": "anomalies (Price==0 & aborted errors)", "Rows": len(anomalies), "Share (%)": round(len(anomalies) / RAW_ROW_COUNT * 100, 2)},
    {"Group": "admin_rows (non-product/shipping/fees)", "Rows": len(admin_rows), "Share (%)": round(len(admin_rows) / RAW_ROW_COUNT * 100, 2)},
    {"Group": "removed_duplicates (overlap + within-sheet)", "Rows": TOTAL_REMOVED_DUPLICATES, "Share (%)": round(TOTAL_REMOVED_DUPLICATES / RAW_ROW_COUNT * 100, 2)},
    {"Group": "TOTAL RECONCILED (Must equal 1,067,371)", "Rows": reconciled_total, "Share (%)": 100.00},
])

print("=" * 70)
print("STEP 9 SUMMARY: ALL VALIDATION ASSERTIONS PASSED")
print("=" * 70)
print("Full Raw-to-Clean Reconciliation Breakdown:")
display(reconciliation_breakdown)

print("\nFinal Row Accounting Table:")
display(accounting_df)

print("\n5 Sample Clean Validated sales Rows (Random Seed = 99):")
display(sales.sample(5, random_state=99))

STEP 9 SUMMARY: ALL VALIDATION ASSERTIONS PASSED
Full Raw-to-Clean Reconciliation Breakdown:


,Group,Rows,Share (%)
0,sales (clean registered),776593,72.76
1,returns (clean registered),17585,1.65
2,guest_transactions (missing Customer ID),227089,21.28
3,anomalies (Price==0 & aborted errors),5968,0.56
4,admin_rows (non-product/shipping/fees),5801,0.54
5,removed_duplicates (overlap + within-sheet),34335,3.22
6,"TOTAL RECONCILED (Must equal 1,067,371)",1067371,100.00



Final Row Accounting Table:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."
6,Step 5: Split Sales / Returns / Anomalies,1027235,1021272,5963,"Partitioned into sales (1,003,357) + returns (..."
7,Step 6: Isolate Missing Customer ID,1021272,794183,227089,"Moved 227,089 guest rows (226,761 sales + 328 ..."
8,Step 7: Quality Checks on sales,794183,794183,0,"Audited top-20 outliers, 0 null descriptions, ..."
9,Step 7B: Isolate Aborted Mega-Order Errors,794183,794178,5,Moved 3 same-day (3-16 min) aborted entry erro...



5 Sample Clean Validated sales Rows (Random Seed = 99):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation,TotalPrice,invoice_date,year_month
51597,494110,21123,SET/10 IVORY SPOTTY PARTY CANDLES,24,2010-01-11 12:29:00,1.25,12549,USA,Year 2009-2010,False,30.00,2010-01-11,2010-01
694182,551137,21754,HOME BUILDING BLOCK WORD,1,2011-04-26 13:05:00,5.95,15581,United Kingdom,Year 2010-2011,False,5.95,2011-04-26,2011-04
124817,501304,79323W,WHITE CHERRY LIGHTS,1,2010-03-15 17:14:00,6.75,17231,United Kingdom,Year 2009-2010,False,6.75,2010-03-15,2010-03
332077,521816,22402,MAGNETS PACK OF 4 VINTAGE COLLAGE,1,2010-09-08 15:30:00,1.25,15555,United Kingdom,Year 2009-2010,False,1.25,2010-09-08,2010-09
639399,546003,22631,CIRCUS PARADE LUNCH BOX,2,2011-03-08 15:07:00,1.95,16712,United Kingdom,Year 2010-2011,False,3.90,2011-03-08,2011-03


## Step 10: Export Cleaned Partitions to Parquet

### Why Parquet Instead of CSV?
1. **Schema & Dtype Fidelity**: CSV discards dtypes (`Customer ID` reverts to `float64`, `Invoice`/`StockCode` revert to mixed `object` types, and `InvoiceDate` turns back into plain text). Parquet preserves exact types (`Int64`, `string`, `datetime64`, `boolean`, `float64`) natively.
2. **Speed & Compression**: Columnar Snappy compression cuts disk footprint by ~5–10x compared to Excel/CSV and loads in fractions of a second for downstream RFM segmentation and 90-day CLV modeling.

In [19]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "sales_clean": sales,
    "returns_clean": returns,
    "guest_transactions": guest_transactions,
    "admin_rows": admin_rows,
}

export_stats = []
for name, df_part in exports.items():
    out_path = PROCESSED_DIR / f"{name}.parquet"
    df_part.to_parquet(out_path, index=False)
    
    # Round-trip verification read
    reloaded = pd.read_parquet(out_path)
    assert reloaded.shape == df_part.shape, f"Round-trip shape mismatch for {name}!"
    
    export_stats.append({
        "dataset": f"{name}.parquet",
        "rows": len(reloaded),
        "cols": reloaded.shape[1],
        "size_mb": round(out_path.stat().st_size / (1024 * 1024), 2),
        "path": str(out_path.relative_to(PROJECT_ROOT)),
    })

export_df = pd.DataFrame(export_stats)

accounting_df = log_row_accounting(
    step="Step 10: Export Parquet Artifacts",
    rows_before=len(sales) + len(returns),
    rows_after=len(sales) + len(returns),
    reason="Exported sales_clean, returns_clean, guest_transactions, and admin_rows to data/processed/*.parquet with round-trip verification",
)

print("=" * 70)
print("STEP 10 SUMMARY: PARQUET EXPORTS VERIFIED")
print("=" * 70)
display(export_df)

print("\nFinal Complete Row Accounting Ledger:")
display(accounting_df)

print("\n5 Sample Rows Reloaded From sales_clean.parquet (Random Seed = 42):")
sales_reloaded = pd.read_parquet(PROCESSED_DIR / "sales_clean.parquet")
display(sales_reloaded.sample(5, random_state=42))

STEP 10 SUMMARY: PARQUET EXPORTS VERIFIED


,dataset,rows,cols,size_mb,path
0,sales_clean.parquet,776593,13,5.92,data\processed\sales_clean.parquet
1,returns_clean.parquet,17585,13,0.33,data\processed\returns_clean.parquet
2,guest_transactions.parquet,227089,14,1.52,data\processed\guest_transactions.parquet
3,admin_rows.parquet,5801,14,0.16,data\processed\admin_rows.parquet



Final Complete Row Accounting Ledger:


,step,rows_before,rows_after,removed,reason
0,Step 0: Load & Concat,1067371,1067371,0,"Loaded Year 2009-2010 (525,461) + Year 2010-20..."
1,Step 1: Type & Text Normalization,1067371,1067371,0,"Cast dtypes (datetime, string, Int64), strippe..."
2,Step 2: Sheet Overlap Audit,1067371,1067371,0,Audited overlap (2010-12-01 to 2010-12-09): 22...
3,Step 3A: Remove Cross-Sheet Overlap,1067371,1044848,22523,Dropped Year 2009-2010 copy of identical Dec 1...
4,Step 3B: Remove Within-Sheet Duplicates,1044848,1033036,11812,Dropped exact full-row duplicates (keep='first...
5,Step 4B: Move Non-Product Rows to admin_rows,1033036,1027235,5801,"Moved 5,801 confirmed admin/non-product rows (..."
6,Step 5: Split Sales / Returns / Anomalies,1027235,1021272,5963,"Partitioned into sales (1,003,357) + returns (..."
7,Step 6: Isolate Missing Customer ID,1021272,794183,227089,"Moved 227,089 guest rows (226,761 sales + 328 ..."
8,Step 7: Quality Checks on sales,794183,794183,0,"Audited top-20 outliers, 0 null descriptions, ..."
9,Step 7B: Isolate Aborted Mega-Order Errors,794183,794178,5,Moved 3 same-day (3-16 min) aborted entry erro...



5 Sample Rows Reloaded From sales_clean.parquet (Random Seed = 42):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancellation,TotalPrice,invoice_date,year_month
757201,579503,22699,ROSES REGENCY TEACUP AND SAUCER,4,2011-11-29 16:23:00,2.95,17097,Spain,Year 2010-2011,False,11.80,2011-11-29,2011-11
458697,546091,84978,HANGING HEART JAR T-LIGHT HOLDER,12,2011-03-09 11:53:00,1.25,14936,Channel Islands,Year 2010-2011,False,15.00,2011-03-09,2011-03
497366,550993,22384,LUNCH BAG PINK POLKADOT,10,2011-04-21 19:51:00,1.65,15810,United Kingdom,Year 2010-2011,False,16.50,2011-04-21,2011-04
164243,510993,21890,S/6 WOODEN SKITTLES IN COTTON BAG,6,2010-06-04 13:47:00,2.95,18206,United Kingdom,Year 2009-2010,False,17.70,2010-06-04,2010-06
274,489488,22086,PAPER CHAIN KIT 50'S CHRISTMAS,5,2009-12-01 10:59:00,2.95,17238,United Kingdom,Year 2009-2010,False,14.75,2009-12-01,2009-12


---

## Cleaning Summary (For Interview Defense & Project README)

### 1. Row Accounting Across All Steps (`1,067,371` Raw Rows Reconciled)

| Step | Action | Rows Before | Rows After | Removed / Partitioned | Outcome |
| :--- | :--- | :---: | :---: | :---: | :--- |
| **Step 0** | Load `Year 2009-2010` (`525,461`) + `Year 2010-2011` (`541,910`) | 1,067,371 | 1,067,371 | 0 | Combined with `source_sheet` provenance |
| **Step 1** | Normalize dtypes & text (`datetime64`, `string`, `Int64`, `is_cancellation`) | 1,067,371 | 1,067,371 | 0 | Fixed mixed `int`/`str` types & 213,035 whitespace diffs |
| **Step 2** | Audit Dec 1–9, 2010 cross-sheet overlap | 1,067,371 | 1,067,371 | 0 | Verified 22,523 rows/sheet match 100% (`0` diffs) |
| **Step 3A** | Remove Sheet 1 copy of cross-sheet overlap | 1,067,371 | 1,044,848 | `-22,523` | Dropped overlap duplicates cleanly |
| **Step 3B** | Remove exact full-row within-sheet duplicates | 1,044,848 | 1,033,036 | `-11,812` | Dropped exact duplicates; kept 4 differing Key4 pairs |
| **Step 4** | Move confirmed non-product/admin rows (`25` codes) | 1,033,036 | 1,027,235 | `-5,801` | Moved to `admin_rows.parquet` |
| **Step 5** | Split into `sales`, `returns`, and `anomalies` | 1,027,235 | 1,021,272 | `-5,963` | `5,963` zero-price / warehouse logs moved to `anomalies` |
| **Step 6** | Isolate missing / zero `Customer ID` rows | 1,021,272 | 794,183 | `-227,089` | Moved `226,761` sales + `328` returns to `guest_transactions.parquet` |
| **Step 7** | Quality checks on `sales` + isolate 3 same-day entry errors | 794,183 | 794,178 | `-5` | Moved 5 aborted product error rows to `anomalies` |
| **Step 8–9** | Add `TotalPrice`, `invoice_date`, `year_month` & loud validation | 794,178 | 794,178 | 0 | **776,593 `sales_clean`** + **17,585 `returns_clean`** verified |

---

### 2. Key Engineering Decisions & Reasoning
1. **Strict Transaction-Level Scope**: Deliberately excluded RFM aggregations, scalers, and log transforms from Phase 2 because selecting a temporal split cutoff ($T_{\text{cutoff}}$) belongs in Feature Engineering; aggregating across all 24 months beforehand causes **temporal data leakage** in 90-day CLV.
2. **Type & Text Normalization Before Deduplication**: Fixed mixed `int`/`str` Excel artifacts (`1,047,871` ints in `Invoice`, `4` ints in `Description`, `3,471` lowercase SKUs, `213,035` trailing spaces) using pandas nullable `string` and `Int64` so genuine `<NA>` nulls never became `"nan"` or `0`.
3. **Overlap Audit Before Deduplication**: Investigated the Dec 1–9, 2010 overlap *before* deduplicating, proving all `22,523` rows per sheet were 100% identical (including `321` within-sheet repeat lines per sheet inside those 9 days) and keeping the `Year 2010-2011` copy.
4. **Data-Driven Non-Product SKU Discovery**: Audited all `61` SKUs outside `^\d{5}[A-Z]*$` (`5,980` rows) rather than blindly applying blog blacklists. Separated `25` administrative/fee codes (`POST`, `DOT`, `C2`, `M`, `D`, `B`, `ADJUST*`, `BANK CHARGES`, `AMAZONFEE`, `CRUK`, `S`, `TEST*`, `GIFT*`, `C3` = `5,801` rows) from `36` legacy physical merchandise SKUs (`DCGS*`, `SP1002`, `PADS` = `179` rows).
5. **Split Before Filtering `Quantity > 0`**: Preserved `17,585` registered `returns_clean` (`-£464,336.83`) alongside `776,593` `sales_clean` (`£16,783,959.54`) so downstream CLV/RFM models can compute net spend and return-rate signals.
6. **Forensic Cancellation Matching on Mega-Orders**: Cross-matched extreme orders on `Customer ID`, `StockCode`, `Quantity`, **`Price`**, and `admin_rows` reversals. Isolated `581483/C581484` (`£168.5k`), `541431/C541433` (`£77.2k`), and `556444` (`£39.0k`, reversed by Manual `C556445` in `admin_rows`), while preserving `C556448` (`-£297`) in `returns` to balance Customer `15098`'s legitimate order `556442`.

---

### 3. Known Limitations (To Document in README & Modeling Phase)
- **Within-Sheet Duplicates (`11,812` rows / `£54,228.42` = `0.29%` of revenue)**: ~85.7% (`10,127` rows) appeared non-consecutively within the same invoice (often `Qty=1`), which could reflect unmerged cart clicks rather than POS double-scans. Dropping exact full-row duplicates enforces strict row uniqueness at a max trade-off of `0.29%` revenue.
- **Guest Checkouts (`227,089` rows / `13.11%` of sales revenue)**: Excluded from registered customer models. Guest row share is non-stationary, spiking during Dec–Jan holiday windows (`37.37%` in Dec 2010, `38.36%` in Jan 2011) vs. `14%–24%` in spring/autumn.
- **Store Holiday Closures**: Two 12-day Christmas/New Year breaks (`2009-12-23 -> 2010-01-04` and `2010-12-23 -> 2011-01-04`) and 5-day Easter closures cause `0`-invoice weeks; Recency calculations must not confuse store closures with customer churn.
- **Odd Country Codes**: Retained `Unspecified` (`518` rows, `6` customers) and `European Community` (`57` rows, `1` customer) as valid registered purchases; geographic features should treat `Unspecified` explicitly.
- **Net Revenue Calculation Required**: Downstream RFM/CLV pipelines must combine `sales_clean` and `returns_clean` (net Monetary = `sales + returns`) so multi-day B2B cancellations (e.g., `+28d`, `+97d` re-invoicing) do not inflate Monetary estimates.